# Chapter 6: Algorithm Pipelines and Stacking

## Overview
This notebook covers constructing robust machine learning workflows using Scikit-Learn Pipelines:
1. **Data Leakage in Preprocessing**: Understanding how improper scaling during cross-validation skews performance estimates.
2. **Building Pipelines**: Encapsulating transformers and estimators using `Pipeline` and `make_pipeline`.
3. **GridSearchCV with Pipelines**: Tuning preprocessing parameters and model hyperparameters simultaneously.
4. **Selecting Algorithms via Grid Search**: Dynamically swapping transformers and estimators within a single optimization pipeline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

# Matplotlib global settings
plt.rc("font", size=10)
plt.rc("axes", labelsize=11, titlesize=12)

## 1. Data Leakage and Preprocessing Risk

A common pitfall in machine learning pipelines occurs when preprocessing transformers (e.g., `StandardScaler`, `PCA`) are fitted on the **entire dataset** prior to cross-validation:

- **Data Leakage**: Test fold statistics (mean, standard deviation, principal components) leak into the training fold during preprocessing.
- **Over-Optimistic Metrics**: Models appear to generalize better on cross-validation than on truly unseen hold-out test sets.
- **Solution**: Wrap transformers and estimators inside a Scikit-Learn `Pipeline` so scaling is computed exclusively on training folds during each cross-validation split.

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- INCORRECT WAY: Preprocessing before cross-validation (Data Leakage) ---
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Scaled using all X_train folds together

# GridSearchCV cross-validation on pre-scaled data leaks fold statistics across splits
grid_leaked = GridSearchCV(SVC(), param_grid={"C": [0.1, 1, 10], "gamma": [0.01, 0.1, 1]}, cv=5)
grid_leaked.fit(X_train_scaled, y_train)

# --- CORRECT WAY: Scaling inside a Pipeline (No Data Leakage) ---
pipe = Pipeline([
    ("scaler", MinMaxScaler()),
    ("svm", SVC())
])

param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": [0.01, 0.1, 1]
}

grid_pipe = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid_pipe.fit(X_train, y_train)

print(f"Incorrect Pre-scaled CV Score (Leaked): {grid_leaked.best_score_*100:.2f}%")
print(f"Correct Pipeline CV Score (Isolated):   {grid_pipe.best_score_*100:.2f}%")
print(f"True Hold-out Test Accuracy:            {grid_pipe.score(X_test, y_test)*100:.2f}%")

## 2. Constructing Pipelines (`Pipeline` vs `make_pipeline`)

Pipelines link transformers and estimators into a single unified object:

- **`Pipeline([('name', Transformer()), ('estimator', Model())])`**: Explicitly names each stage.
- **`make_pipeline(Transformer(), Model())`**: Automatically generates stage names using lowercased class names (e.g., `standardscaler`, `svc`).

In [ ]:
# Explicit Pipeline construction
pipe_explicit = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("classifier", LogisticRegression())
])

# Convenience shortcut via make_pipeline
pipe_short = make_pipeline(StandardScaler(), PCA(n_components=2), LogisticRegression())

print("Explicit Pipeline Stage Names:", list(pipe_explicit.named_steps.keys()))
print("Short Pipeline Stage Names:   ", list(pipe_short.named_steps.keys()))

# Fit and evaluate pipeline on raw input data directly
pipe_short.fit(X_train, y_train)
print(f"Pipeline Test Accuracy: {pipe_short.score(X_test, y_test)*100:.2f}%")

## 3. Grid Search Over Preprocessing Steps & Model Parameters

Combining `Pipeline` with `GridSearchCV` allows simultaneous optimization of preprocessing parameters (e.g., PCA component count) and model parameters (e.g., SVM regularizer $C$).

Parameter grids reference specific pipeline steps using the double-underscore syntax: `<step_name>__<parameter_name>`.

In [ ]:
pipe_full = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("svm", SVC())
])

# Define grid covering both PCA components and SVM parameters
param_grid_full = {
    "pca__n_components": [2, 5, 10, 15],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1]
}

grid_full = GridSearchCV(pipe_full, param_grid=param_grid_full, cv=5, n_jobs=-1)
grid_full.fit(X_train, y_train)

print(f"Optimal PCA Components:   {grid_full.best_params_['pca__n_components']}")
print(f"Optimal SVM Parameters:   C={grid_full.best_params_['svm__C']}, gamma={grid_full.best_params_['svm__gamma']}")
print(f"Best Cross-Val Accuracy:  {grid_full.best_score_*100:.2f}%")
print(f"Hold-out Test Accuracy:   {grid_full.score(X_test, y_test)*100:.2f}%")

## 4. Grid Searching Over Pipeline Class Architecture

Pipelines can search across different preprocessing algorithms and model families by assigning estimator and transformer objects directly as parameters in `GridSearchCV`.

In [ ]:
# Base pipeline template
pipe_search = Pipeline([
    ("preprocessing", StandardScaler()),
    ("classifier", SVC())
])

# Define alternate model architectures and preprocessors
param_grid_architecture = [
    # Search path 1: SVM with StandardScaler or MinMaxScaler
    {
        "preprocessing": [StandardScaler(), MinMaxScaler()],
        "classifier": [SVC()],
        "classifier__C": [0.1, 1, 10],
        "classifier__gamma": [0.01, 0.1]
    },
    # Search path 2: Random Forest (No scaling required)
    {
        "preprocessing": [None],
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [50, 100],
        "classifier__max_depth": [3, 5, None]
    }
]

grid_arch = GridSearchCV(pipe_search, param_grid=param_grid_architecture, cv=5, n_jobs=-1)
grid_arch.fit(X_train, y_train)

print(f"Selected Preprocessor: {grid_arch.best_params_['preprocessing']}")
print(f"Selected Classifier:   {grid_arch.best_params_['classifier']}")
print(f"Best CV Accuracy:      {grid_arch.best_score_*100:.2f}%")
print(f"Test Set Accuracy:     {grid_arch.score(X_test, y_test)*100:.2f}%")